# Add Historical Movies

This notebook allows you to add movies that you watched **before you started tracking**.
These movies will be added at the **beginning** of the dataframe (low index) **without viewing dates or comments**.

## Imports

In [1]:
import polars as pl
from polars import col as c
import os, sys, requests, re
from dotenv import load_dotenv, find_dotenv
from tqdm import tqdm
from datetime import datetime
import subprocess

load_dotenv(find_dotenv())

True

## Constants

In [2]:
OMDB_API_KEY = os.getenv("OMDB_API_KEY")
FILES_DIR = "../data"

## Load DataFrame

In [3]:
file_name = "movies_df.parquet"
file_path = os.path.join(FILES_DIR, file_name)
movies_df = pl.read_parquet(file_path)
print(f"Loaded {len(movies_df)} movies")
movies_df.head()

Loaded 671 movies


index,title,year,viewed,liked,omdb_id,genre,director,country,actors,box_office,writer,language,imdb_rating,comment
u32,str,i64,date,bool,str,str,str,str,str,i32,str,str,f32,str
1,"""Sister Act""",1992,null,false,"""tt0105417""","""Comedy, Family, Music""","""Emile Ardolino""","""United States""","""Whoopi Goldberg, Maggie Smith,…",139605150,"""Paul Rudnick""","""English""",6.5,""""""
2,"""Summer Lovers""",1982,null,false,"""tt0084737""","""Comedy, Drama, Romance""","""Randal Kleiser""","""United States""","""Peter Gallagher, Daryl Hannah,…",4968000,"""Randal Kleiser""","""English""",5.8,""""""
3,"""John Wick""",2014,null,false,"""tt2911666""","""Action, Crime, Thriller""","""Chad Stahelski""","""United States""","""Keanu Reeves, Michael Nyqvist,…",43037835,"""Derek Kolstad""","""English, Russian, Hungarian""",7.5,""""""
4,"""Memento""",2000,null,false,"""tt0209144""","""Drama, Mystery, Thriller""","""Christopher Nolan""","""United States""","""Guy Pearce, Carrie-Anne Moss, …",25544867,"""Christopher Nolan, Jonathan No…","""English""",8.4,""""""
5,"""Bicycle Thieves""",1948,null,false,"""tt0040522""","""Drama""","""Vittorio De Sica""","""Italy""","""Lamberto Maggiorani, Enzo Stai…",371111,"""Cesare Zavattini, Luigi Bartol…","""Italian, German""",8.2,""""""


## Helper Functions

In [ ]:
def fetch_english_title(omdb_id):
    """Fetch movie data from OMDb API by IMDb ID."""
    try:
        url = f"http://www.omdbapi.com/?apikey={OMDB_API_KEY}&i={requests.utils.quote(omdb_id)}"
        response = requests.get(url)
        data = response.json()
        if data.get("Response") == "True" and "Title" in data:
            return data
        else:
            print(f"Not found in OMDb: {omdb_id}")
            return "Not found"
    except Exception as e:
        print(f"OMDb error for {omdb_id}: {e}")
        return "Not found"


def add_historical_movie(
    df: pl.DataFrame,
    omdb_id: str,
    liked: bool = False,
) -> pl.DataFrame:
    """
    Add a historical movie to the BEGINNING of the dataframe.
    No viewing date or comment is added (viewed = None).

    Args:
        df (pl.DataFrame): The dataframe to add the row to.
        omdb_id (str): The IMDb ID (e.g., 'tt0111161').
        liked (bool): Whether you liked the movie (default: False).

    Returns:
        pl.DataFrame: DataFrame with the new row at the beginning.
    """
    # Check if movie already exists
    if omdb_id in df["omdb_id"].to_list():
        print(f"⚠️  Movie {omdb_id} already exists in the database!")
        return df

    # Fetch movie data from OMDb
    creds = fetch_english_title(omdb_id)
    if creds == "Not found":
        print(f"❌ ERROR: Cannot fetch data for {omdb_id}")
        raise ValueError("CredsNotFoundError")

    # Prepare new row
    new_row = {col: None for col in df.columns}
    new_row["omdb_id"] = omdb_id
    new_row["liked"] = liked
    new_row["viewed"] = None  # No viewing date for historical movies
    new_row["comment"] = None  # No comment

    # Fill movie metadata from OMDb
    for col_name in [
        "title",
        "year",
        "genre",
        "director",
        "country",
        "actors",
        "box_office",
        "writer",
        "language",
        "imdb_rating",
    ]:
        omdb_key = "".join(
            [
                word.capitalize() if word != "imdb" else word
                for word in col_name.split("_")
            ]
        )
        new_row[col_name] = creds.get(omdb_key)

        # Handle box_office conversion
        if col_name == "box_office" and new_row[col_name]:
            try:
                new_row[col_name] = int(re.sub(r"[^\d]", "", new_row[col_name]))
            except:
                print(f"⚠️  Box office value is {new_row[col_name]}. Setting to null.")
                new_row[col_name] = None

    # Create single-row DataFrame
    new_row_df = pl.DataFrame([new_row])
    new_row_df = new_row_df.cast(df.schema)

    # Add at the BEGINNING (before all other movies)
    result_df = pl.concat([new_row_df, df], how="vertical")

    # Reindex: assign sequential indices starting from 1
    result_df = result_df.with_columns(
        pl.arange(1, len(result_df) + 1).cast(pl.UInt32).alias("index")
    )

    print(f"✅ Added '{creds['Title']}' ({creds['Year']}) at the beginning")
    return result_df


# Test function
print("\nTest - adding She Led Two Lives:")
test_df = add_historical_movie(
    df=movies_df,
    omdb_id="tt0111162",
    liked=True,
)
print(f"\nFirst 3 movies after test:")
display(test_df.head(3))


Test - adding She Led Two Lives:
⚠️  Box office value is N/A. Setting to null.
✅ Added 'She Led Two Lives' (1994) at the beginning

First 3 movies after test:


index,title,year,viewed,liked,omdb_id,genre,director,country,actors,box_office,writer,language,imdb_rating,comment
u32,str,i64,date,bool,str,str,str,str,str,i32,str,str,f32,str
1,"""She Led Two Lives""",1994,null,true,"""tt0111162""","""Drama""","""Bill Corcoran""","""United States""","""Connie Sellecca, Perry King, A…",null,"""Kathleen Rowell""","""English""",5.1,null
2,"""Sister Act""",1992,null,false,"""tt0105417""","""Comedy, Family, Music""","""Emile Ardolino""","""United States""","""Whoopi Goldberg, Maggie Smith,…",139605150,"""Paul Rudnick""","""English""",6.5,""""""
3,"""Summer Lovers""",1982,null,false,"""tt0084737""","""Comedy, Drama, Romance""","""Randal Kleiser""","""United States""","""Peter Gallagher, Daryl Hannah,…",4968000,"""Randal Kleiser""","""English""",5.8,""""""


## Add Historical Movie

Set the `NEW_MOVIE_OMDB_ID` to the IMDb ID of the movie you want to add.
Set `liked=True` if you liked the movie.

**Note:** The movie will be added at the beginning without a viewing date.

In [7]:
NEW_MOVIE_OMDB_ID = "tt0111161"  # Example: The Shawshank Redemption
LIKED = True

In [8]:
# Check if movie already exists
if NEW_MOVIE_OMDB_ID in movies_df["omdb_id"].to_list():
    print(f"⚠️  Movie {NEW_MOVIE_OMDB_ID} already exists!")
else:
    # Preview what will be added
    print("Preview - this movie will be added at the beginning:")
    display(
        add_historical_movie(
            df=movies_df,
            omdb_id=NEW_MOVIE_OMDB_ID,
            liked=LIKED,
        ).head(3)
    )

⚠️  Movie tt0111161 already exists!


In [ ]:
# Actually add the movie
print(f"Initial dataframe size: {len(movies_df)} movies")

movies_df = add_historical_movie(
    df=movies_df,
    omdb_id=NEW_MOVIE_OMDB_ID,
    liked=LIKED,
)

print(f"After adding: {len(movies_df)} movies")
display(movies_df.head())

## Bulk Add Multiple Historical Movies

If you have multiple movies to add at once, use this section.

In [ ]:
# List of historical movies to add (IMDb ID, liked)
HISTORICAL_MOVIES = [
    # ("tt0111161", True),   # The Shawshank Redemption
    # ("tt0071562", True),   # The Godfather Part II
    # ("tt0468569", True),   # The Dark Knight
    # Add more movies here...
]

if HISTORICAL_MOVIES:
    print(f"Adding {len(HISTORICAL_MOVIES)} historical movies...\n")
    for omdb_id, liked in tqdm(HISTORICAL_MOVIES):
        try:
            movies_df = add_historical_movie(
                df=movies_df,
                omdb_id=omdb_id,
                liked=liked,
            )
        except Exception as e:
            print(f"Failed to add {omdb_id}: {e}")
            continue

    print(f"\n✅ Bulk add complete. Total movies: {len(movies_df)}")
else:
    print("No movies to add in bulk mode.")

Adding 4 historical movies...



  0%|          | 0/4 [00:00<?, ?it/s]

⚠️  Movie tt0068646 already exists in the database!
⚠️  Movie tt0050083 already exists in the database!
⚠️  Movie tt0095016 already exists in the database!


100%|██████████| 4/4 [00:00<00:00, 18.22it/s]

Failed to add tt2665218: conversion from `str` to `i64` failed in column 'year' for 1 out of 1 values: ["2002–"]

✅ Bulk add complete. Total movies: 673


## Save DataFrame

In [26]:
# Save to parquet
file_name = "movies_df.parquet"
file_path = os.path.join(FILES_DIR, file_name)
movies_df.write_parquet(file_path)
print(f"✅ Saved to {file_path}")

✅ Saved to ../data/movies_df.parquet


## Commit and Push Changes

In [28]:
# Git commit and push
try:
    subprocess.run(["git", "add", "-f", file_path], check=True)
    print(f"Added {file_path} to staging.")

    commit_message = "add historical movie"
    subprocess.run(["git", "commit", "-m", commit_message], check=True)
    print(f"Committed changes with message: '{commit_message}'")

    subprocess.run(["git", "push"], check=True)
    print(f"Pushed changes to remote.")

except subprocess.CalledProcessError as e:
    print(f"Git command failed: {e}")
except FileNotFoundError:
    print("Git command not found. Ensure Git is installed and in your PATH.")

Added ../data/movies_df.parquet to staging.
[add-watched-movies-long-time-ago c851fdb] add historical movie
 1 file changed, 0 insertions(+), 0 deletions(-)
Committed changes with message: 'add historical movie'
Pushed changes to remote.


To https://github.com/KonstantinBurkin/movie-list.git
   a9587a8..c851fdb  add-watched-movies-long-time-ago -> add-watched-movies-long-time-ago


## Optional: Export to Excel

In [ ]:
# Uncomment to export to Excel
# file_name = "movies_df.xlsx"
# file_path = os.path.join(FILES_DIR, file_name)
# movies_df.write_excel(file_path)
# print(f"Exported to {file_path}")